In [2]:
def main():
    
    # imports
    import pandas as pd # data manipulation/access
    import utils # custom utility functions
    from IPython.display import display, HTML # hyperlink access
    #import warnings
    #warnings.simplefilter(action='ignore', category=FutureWarning)
    pd.set_option('display.max_columns', None)
    #import fuzzywuzzy # fuzzy match
    #pd.set_option('display.max_colwidth', None)
    #import re # regex

    dimension_1 = ['Diffuse or Multifocal','Symmetric','Frontal vs Posterior Predominance','Telltale Grey Matter Involvement','Cortical Involvement/Subcortical Lesions','U-Fiber/Juxtacortical Involvement', 'Ventriculomegaly vs Hydrocephalus']
    dimension_2 = ['Spinal Cord Involvement','Periventricular Involvement']
    dimension_3 = ['Subcortical Structures']

    # main loop
    flag = True
    while flag:

        # data
        findings_df = pd.read_excel('hsp_data.xlsx').iloc[:, list(range(0,20))].fillna('')
        df = pd.read_excel('hsp_data.xlsx').fillna('')     

        # initially, additional_category is set to True to start a search
        # if set to False, return results with the only selected search category
        additional_category_flag = True

        # keep track of all choices made by the user
        all_choices = []
        while additional_category_flag:
            
            # display list of main categories
            access_categories = list(utils.findings_categorized.keys())
            print("List of categories:")
            print("*******************\n")
            print('\n\n'.join(access_categories))
            print()
            print()

            # user input for main category #
            category = input("Enter name of MRI finding: ")
            
            # keyboard interrupt
            if category == "":
                return "Forced exit."
            
            # invalid input handling #
            while category not in access_categories:
                print()
                category = input("Invalid input. Choose a category from the list above (case sensitive): ")
            
            print()
            print("- You chose: [", category,"].")
            print()
            print()
            
            # check if main category is more than 1 dimension
            if category not in dimension_1:

                # 3 dimension
                if category not in dimension_2:

                    # display list of subcategories
                    access_subcategories = utils.findings_categorized['Subcortical Structures']
                    print("List of subcategories:")
                    print("**********************\n")
                    print('\n\n'.join(list(access_subcategories)))
                    print()
                    print()

                    # user input for subcategory #
                    subcategory = input("Choose a subcategory: ")

                    # keyboard interrupt
                    if subcategory == "":
                        return "Forced exit."
                    
                    # invalid input handling #
                    while subcategory not in access_subcategories:
                        print()
                        subcategory = input("Invalid input. Choose a subcategory from the list above (case sensitive): ")

                    print()
                    print("- You chose: [", subcategory,"].")
                    print()
                    print()

                    # display list of sub-subcategories
                    access_sub_subcategories = access_subcategories[subcategory]
                    print("List of sub_subcategories:")
                    print("**************************\n")
                    print('\n\n'.join(list(access_sub_subcategories)))
                    print()
                    print()

                    # user input for sub_subcategory #
                    sub_subcategory = input("Choose a sub-subcategory: ")

                    # keyboard interrupt
                    if sub_subcategory == "":
                        return "Forced exit."
                    
                    # invalid input handling #
                    while sub_subcategory not in access_sub_subcategories:
                        print()
                        sub_subcategory = input("Invalid input. Choose a subcategory from the list above (case sensitive): ")
                        
                    print()
                    print("- You chose: [", sub_subcategory,"].")

                    # column title
                    col_title = str(category+':'+subcategory+':'+sub_subcategory)

                # 2 dimension
                else:

                    # display list of subcategories
                    access_subcategories = utils.findings_categorized[category]
                    print("\n\nList of subcategories:")
                    print("**********************\n")
                    print('\n\n'.join(list(access_subcategories)))
                    print()
                    print()

                    # user input for subcategory #
                    subcategory = input("Choose a subcategory: ")

                    # keyboard interrupt
                    if subcategory == "":
                        return "Forced exit."
                    
                    # invalid input handling #
                    while subcategory not in access_subcategories:
                        print()
                        subcategory = input("Invalid input. Choose a subcategory from the list above (case sensitive): ")

                    print()
                    print("- You chose: [", subcategory,"].")
                    print()
                    print()

                    # column title
                    col_title = str(category+':'+subcategory)

            # 1 dimension        
            else:

                # column title
                col_title = category
            print()
            print()

            # verify all unique values are selectable
            column_values = list(df[col_title].unique())
            access_values_of_interest = []
            for column_value in column_values:
                values = column_value.split(',')
                for value in values:
                    if str(value.strip()) not in access_values_of_interest:
                        access_values_of_interest.append(value.strip())
                        
            # display list of values of interest
            print("List of Values of Interest:")
            print("***************************\n")
            access_values_of_interest.remove('') if ('' in access_values_of_interest) else None
            print('\n\n'.join(access_values_of_interest))
            print()

            # user input for value of interest #
            value_of_interest = input("Choose a value of interest: ")

            # keyboard interrupt
            if value_of_interest == "":
                return "Forced exit."
            
            # invalid input handling #
            while value_of_interest not in access_values_of_interest:
                print()
                value_of_interest = input("Invalid input. Choose a value of interest from the list above (case sensitive): ")

            print()
            print("- You chose: [", value_of_interest,"].")

            # keep relevant rows
            findings_df = findings_df[findings_df[col_title].str.contains(pat=r'\b{}\b'.format(value_of_interest), regex=True)]

            # update list of user choices #############################################################
            all_choices.append(str(col_title+"%"+value_of_interest))
            
            # prompt user to add additional search constraint #
            add_a_category = input("\n\nWould you to add another category? ")

            # keyboard interrupt
            if add_a_category == "":
                return "Forced exit."

            if add_a_category == '0' or add_a_category.lower()=="no":
                additional_category_flag = False
        print()
        print()
        
        # move "Gene" to first column
        current_columns = df.columns.tolist()
        df = df[['Gene'] + [col for col in current_columns if col != 'Gene']]
        
        # highlight relevant rows
        mask = df.index.isin(findings_df.index.tolist())
        result_rows = findings_df.index.tolist()
        def highlight_rows(row):
            if row.name in result_rows:
                return ['background-color: steelblue'] * len(row)
            else:
                return [''] * len(row)

        # display main results
        result_to_display = pd.DataFrame(df.loc[result_rows]).drop(["Clinical Synopsis"], axis=1)
        results = result_to_display.style.apply(highlight_rows, axis=1)
        display(results)
        print()

        # display rest of df ########################
        rest_of_data = df[~mask]
        rest_of_data["Rank"] = 0
        for i in range(len(all_choices)):
            col = all_choices[i].split('%')[0]
            value = all_choices[i].split('%')[1]
            rest_of_data.loc[rest_of_data[col]==value, 'Rank'] += 1
        rest_of_data = pd.DataFrame(rest_of_data.sort_values(by='Rank', ascending=False))
        display(pd.DataFrame(rest_of_data))

        # prompt user to start new search #
        #print(all_choices)
        terminate = input("Would you like to search again? ")
        if terminate == '0' or terminate.lower() == 'no' or terminate == "":
            flag = False

    # exit message
    print("Goodbye!")

In [ ]:
main()